# recount2 CLAMP models with all pathways prior (all samples)

**Environment:** `clamp-analyses`

Runs CLAMPfull with all pathways prior (Hallmark, Reactome, GO CC, C8) using all recount2 samples, reusing the FBM, SVD, and CLAMPbase results already generated in `nbs/01_model_building/03_recount2/00_recount2.ipynb`.

Steps:
1. Load existing FBM, SVD, CLAMPbase, and CLAMP_K from `config$recount2$DATASET_FOLDER`
2. Run CLAMPfull with the combined all-pathways prior
3. Save results to `config$recount2$DATASET_FOLDER/CLAMPfull_hall`

## Load libraries

In [ ]:
library(bigstatsr)
library(data.table)
library(dplyr)
library(Matrix)
library(here)
library(CLAMP)

source(here("config.R"))

## Configuration

In [ ]:
output_data_dir <- config$recount2$DATASET_FOLDER
pathways_path   <- here::here('data/pathways')

MULTIPLIER <- 100
MAX_ITER   <- 5000

message("recount2 output dir: ", output_data_dir)

## Load pre-built recount2 inputs

In [ ]:
recount2_genes    <- readRDS(file.path(output_data_dir, "recount2_genes.rds"))
samples           <- readRDS(file.path(output_data_dir, "recount2_samples.rds"))
recount2_fbm_filt <- readRDS(file.path(output_data_dir, "recount2_fbm_filt.rds"))
recount2_svdRes   <- readRDS(file.path(output_data_dir, "recount2_svdRes.rds"))
recount2_baseRes  <- readRDS(file.path(output_data_dir, "CLAMPbase.rds"))
CLAMP_K_recount2  <- readRDS(file.path(output_data_dir, "CLAMP_K_recount2.rds"))

message("Genes: ", length(recount2_genes))
message("Samples: ", length(samples))
message("CLAMP K: ", CLAMP_K_recount2)

## Load and match all pathways prior

In [ ]:
hall_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "h.all.v2026.1.Hs.symbols.gmt"))
reactome_gmt <- CLAMP:::read_gmt(file.path(pathways_path, "c2.cp.reactome.v2026.1.Hs.symbols.gmt"))
gocc_gmt     <- CLAMP:::read_gmt(file.path(pathways_path, "c5.go.cc.v2026.1.Hs.symbols.gmt"))
c8_gmt       <- CLAMP:::read_gmt(file.path(pathways_path, "c8.all.v2026.1.Hs.symbols.gmt"))

names(hall_gmt)     <- paste0("HALL_",     names(hall_gmt))
names(reactome_gmt) <- paste0("REACTOME_", names(reactome_gmt))
names(gocc_gmt)     <- paste0("GOCC_",     names(gocc_gmt))
names(c8_gmt)       <- paste0("C8_",       names(c8_gmt))

all_pathways_list <- list(
  HALL     = hall_gmt,
  REACTOME = reactome_gmt,
  GOCC     = gocc_gmt,
  C8       = c8_gmt
)

all_pathways_pathMat <- gmtListToSparseMat(all_pathways_list)
all_pathways_matched <- getMatchedPathwayMat(all_pathways_pathMat, recount2_genes)
message("Loaded and matched all pathways matrix against recount2 genes")

## Run CLAMPfull with all pathways prior

In [ ]:
message("Running CLAMPfull with all pathways prior on recount2 (all samples)...")

recount2_fullRes_hall <- CLAMPfull(
  Y                 = recount2_fbm_filt,
  svdres            = recount2_svdRes,
  priorMat          = all_pathways_matched,
  clamp.base.result = recount2_baseRes,
  use_cpp           = TRUE,
  trace             = TRUE,
  multiplier        = MULTIPLIER,
  max.iter          = MAX_ITER,
  clamp_k           = CLAMP_K_recount2
)

recount2_fullRes_hall$Z <- data.frame(recount2_fullRes_hall$Z)
rownames(recount2_fullRes_hall$Z) <- recount2_genes

recount2_fullRes_hall$B <- data.frame(recount2_fullRes_hall$B)
colnames(recount2_fullRes_hall$B) <- samples

recount2_fullRes_hall$summary <- recount2_fullRes_hall$summary %>%
  dplyr::rename(LV = LV_index) %>%
  dplyr::mutate(LV = paste0('LV', LV))

message("CLAMPfull completed")

## Save results

In [ ]:
dst_dir <- file.path(output_data_dir, "CLAMPfull_hall")
dir.create(dst_dir, showWarnings = FALSE, recursive = TRUE)

saveRDS(recount2_fullRes_hall, file = file.path(output_data_dir, "CLAMPfull_hall.rds"))

write.csv(recount2_fullRes_hall$B,       file.path(dst_dir, "B.csv"))
write.csv(recount2_fullRes_hall$Z,       file.path(dst_dir, "Z.csv"))
write.csv(recount2_fullRes_hall$summary, file.path(dst_dir, "summary.csv"))

message("Results saved to: ", dst_dir)